<a href="https://colab.research.google.com/github/kuds/mesozoic-labs/blob/main/notebooks/jax_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JAX/MJX Dinosaur Training (Colab)

Train a dinosaur species using **MuJoCo MJX** (JAX-accelerated physics) with a
from-scratch PPO implementation in pure JAX. MJX vectorises thousands of parallel
simulations on a single GPU, giving 10-100x speedups over CPU-based Gymnasium training.

**Requirements:** Colab GPU runtime (A100 recommended).

**Supported Species:**
- `trex` — T-Rex: balance → locomotion → bite
- `velociraptor` — Raptor: balance → locomotion → strike
- `brachiosaurus` — Brachio: balance → locomotion → food reach

Set `SPECIES` in the configuration cell below to choose.

In [ ]:
# Install dependencies and verify GPU
!pip install mujoco mujoco-mjx "jax[cuda12]" flax optax

import os
import subprocess

if subprocess.run("nvidia-smi").returncode:
    raise RuntimeError("GPU not found. Use a GPU Colab runtime.")

NVIDIA_ICD_CONFIG_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
    with open(NVIDIA_ICD_CONFIG_PATH, "w") as f:
        f.write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')

os.environ["MUJOCO_GL"] = "egl"

import jax
import mujoco
from mujoco import mjx

print(f"JAX devices: {jax.devices()}")
print(f"MuJoCo: {mujoco.__version__}")
print("Setup complete.")

In [ ]:
# GPU diagnostics — run this anytime to check utilization
# (especially useful to run from a second terminal during training)
import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,utilization.gpu,utilization.memory,memory.used,memory.total,temperature.gpu,power.draw'],
    capture_output=True, text=True,
    env={**__import__('os').environ, 'COLUMNS': '200'},
)
if result.returncode == 0:
    lines = result.stdout.strip().split('\n')
    for line in lines:
        parts = [p.strip() for p in line.split(',')]
        if len(parts) >= 7:
            print(f'GPU:         {parts[0]}')
            print(f'Utilization: {parts[1]} (compute)  {parts[2]} (memory)')
            print(f'Memory:      {parts[3]} / {parts[4]}')
            print(f'Temperature: {parts[5]}   Power: {parts[6]}')
        else:
            print(line)
else:
    print('nvidia-smi failed:', result.stderr)

# Tip: You can also run this from a Colab terminal:
#   watch -n 2 nvidia-smi


In [ ]:
# Clone mesozoic-labs and install with JAX extras
!git clone https://github.com/kuds/mesozoic-labs.git /content/mesozoic-labs 2>/dev/null || echo 'Already cloned'
!pip install -e "/content/mesozoic-labs[jax]" -q

# Ensure the repo root is on sys.path so `from environments.…` works
# even if the editable install did not fully resolve.
import sys
if '/content/mesozoic-labs' not in sys.path:
    sys.path.insert(0, '/content/mesozoic-labs')

from IPython.display import clear_output

clear_output()
print("mesozoic-labs[jax] installed.")

In [ ]:
import logging
import time

# Silence JAX compilation spam (tracing/compiling/XLA warnings every update)
logging.getLogger("jax._src.dispatch").setLevel(logging.ERROR)
logging.getLogger("jax._src.interpreters.pxla").setLevel(logging.ERROR)

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import mujoco
import numpy as np
import optax
from mujoco import mjx

# Shared modules from mesozoic-labs
from environments.shared.jax_reward_termination import (
    compute_total_reward,
    compute_reward_components,
    is_terminated as is_terminated_fn,
)
from environments.shared.jax_ppo import (
    PPOConfig,
    make_actor_critic,
    make_optimizer,
    sample_action,
    compute_gae,
    ppo_loss,
)
from environments.shared.jax_normalization import RunningMeanStd, normalize_obs, update_running_stats, decay_running_stats
from environments.shared.jax_eval import EvalConfig, evaluate_policy_cpu, check_stage_gate
from environments.shared.jax_viz import plot_training_curves, plot_locomotion_diagnostics, record_training_video, create_frame_collage
from environments.shared.mjx_utils import scale_action_jax
from environments.shared.mjx_env import MJXDinoEnv
from environments.shared.obs_functions import SensorLayout, build_bipedal_obs

print(f"JAX {jax.__version__}, devices: {jax.devices()}")
print(f"MuJoCo {mujoco.__version__}")

# Verify GPU
assert jax.devices()[0].platform == "gpu", "No GPU detected — JAX is using CPU. Check runtime type."
print("GPU OK")

In [ ]:
# ============================================================
# USER CONFIGURATION
# ============================================================
SPECIES = "trex"  # Choose: "trex", "velociraptor", "brachiosaurus"
CURRENT_STAGE = 1  # Curriculum stage: 1=balance, 2=locomotion, 3=species-specific
USE_GOOGLE_DRIVE = True  # Set to True to save outputs to Google Drive (persistent across sessions)
VERBOSE = 1  # 0=eval/summary only, 1=periodic updates (default), 2=every update

# Resume from a previous checkpoint (set to a .pkl path, or None to start fresh)
RESUME_FROM = None  # e.g. "checkpoint_100.pkl"

# ============================================================
# Setup via shared library (replaces ~80 lines of hardcoded config)
# ============================================================
from environments.shared.jax_notebook import (
    setup_species,
    setup_output_dirs,
    create_env,
    make_obs_fn,
    make_scale_action_fn,
    make_reward_fns,
    print_species_summary,
)

ctx = setup_species(SPECIES, stage=CURRENT_STAGE)

# Storage directories
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    _STORAGE_ROOT = "/content/drive/MyDrive/mesozoic-labs/logs"
else:
    _STORAGE_ROOT = "logs"

_dirs = setup_output_dirs(SPECIES, CURRENT_STAGE, storage_root=_STORAGE_ROOT)
RUN_DIR = _dirs["run_dir"]
STAGE_DIR = _dirs["stage_dir"]
MODEL_DIR = _dirs["model_dir"]
OUTPUT_DIR = STAGE_DIR  # backward compat

print_species_summary(ctx)
print(f"Run dir:    {RUN_DIR}")
print(f"Stage dir:  {STAGE_DIR}")
print(f"Drive:      {USE_GOOGLE_DRIVE}")
if RESUME_FROM:
    print(f"Resuming from checkpoint: {RESUME_FROM}")

## 1. Load Model into MJX

In [ ]:
# Model already loaded by setup_species() — just display info
mj_model = ctx.mj_model
print(f"{SPECIES.title()} model loaded:")
print(f"  Bodies: {mj_model.nbody}, Joints: {mj_model.njnt}, Actuators: {mj_model.nu}")
print(f"  qpos: {mj_model.nq}, qvel: {mj_model.nv}, Geoms: {mj_model.ngeom}")
print(f"  Total mass: {sum(mj_model.body_mass):.2f} kg, Timestep: {mj_model.opt.timestep * 1000:.1f} ms")

## 2. Environment Functions
Pure-JAX functions for observation, reward, reset, and step. These now use
the **shared modules** from `environments.shared` so that the same reward and
observation logic is used by both the Gymnasium (SB3) and MJX (JAX) training
paths.

**Note:** The shared `reward_functions` use `float()` / Python `if` which break
`jax.vmap` tracing. The reward and termination functions below are re-implemented
in pure JAX (`jnp` only) for vmap compatibility.

In [ ]:
# ---------- Constants (resolved by setup_species) ----------
# All body/site IDs, sensor layout, termination checks, and TOML config
# are now in ctx (SpeciesContext). Extract frequently-used values for
# backward compatibility with downstream cells.

_stage_cfg = ctx.stage_config
_jax_kw = ctx.jax_kwargs
reward_cfg = ctx.reward_cfg
SENSOR_LAYOUT = ctx.sensor_layout
ROOT_BODY_ID = ctx.root_body_id
OBS_DIM = ctx.obs_dim
ACT_DIM = ctx.act_dim
CTRL_RANGE = ctx.ctrl_range
NUM_ENVS = _jax_kw.get("num_envs", 2048)
FRAME_SKIP = ctx.frame_skip
MAX_EPISODE_STEPS = ctx.max_episode_steps
HEALTHY_Z_MIN, HEALTHY_Z_MAX = ctx.healthy_z_range

print(f"Obs dim: {OBS_DIM}, Act dim: {ACT_DIM}")
print(f"Termination: {len(ctx.termination_body_checks)} body, {len(ctx.termination_site_checks)} site checks")
print(f"Reward keys: {[k for k, v in reward_cfg.items() if isinstance(v, (int, float)) and v != 0]}")

In [ ]:
# Observation and action functions (from library)
get_obs = make_obs_fn(ctx)
scale_action = make_scale_action_fn(ctx)
compute_reward, compute_reward_detailed, is_terminated = make_reward_fns(ctx)

print(f"Observation dim: {OBS_DIM}")
print(f"Action dim: {ACT_DIM}")

## 3. Batched MJX Step

A single `jax.jit`-compiled function that steps `N` parallel environments.

In [ ]:
# Create MJXDinoEnv via library helper (applies solver tuning + env_kwargs merge)
env = create_env(ctx, num_envs=NUM_ENVS)
mjx_model = env.mjx_model

print(f"MJXDinoEnv created: {SPECIES} stage {CURRENT_STAGE}")
print(f"  num_envs={NUM_ENVS}, action_dim={env.action_dim}, frame_skip={env.config.frame_skip}")
print(f"  fall_penalty={env.config.fall_penalty}, max_steps={env.config.max_episode_steps}")
print(f"  reward keys: {list(env.config.reward_weights.keys())}")

## 4. Policy Network (Flax)
Uses the shared `ActorCritic` from `environments.shared.jax_ppo`.

In [ ]:
# Initialize using the shared ActorCritic from jax_ppo
network = make_actor_critic(action_dim=ACT_DIM)
rng = jax.random.PRNGKey(42)
dummy_obs = jnp.zeros((OBS_DIM,))
params = network.init(rng, dummy_obs)

# Observation normalization (stabilizes training across species/stages)
obs_rms = RunningMeanStd.create(OBS_DIM)

# Resume from checkpoint if specified
_resume_update = 0
_jax_kw_resume = {}
try:
    from environments.shared.config import load_stage_config as _lsc
    _jax_kw_resume = _lsc(SPECIES, CURRENT_STAGE).get("jax_kwargs", {})
except Exception:
    pass

if RESUME_FROM:
    import pickle
    _ckpt_path = OUTPUT_DIR / RESUME_FROM if not Path(RESUME_FROM).is_absolute() else Path(RESUME_FROM)
    with open(_ckpt_path, "rb") as f:
        _ckpt = pickle.load(f)
    params = jax.device_put(_ckpt["params"])
    _resume_update = _ckpt.get("update", 0)
    if "obs_rms" in _ckpt:
        obs_rms = _ckpt["obs_rms"]
        # When transitioning between stages, the obs distribution shifts
        # (e.g. near-zero velocity in balance → sustained velocity in
        # locomotion).  With 2048 envs the prior count can be ~65M, making
        # update_running_stats nearly a no-op.  Decay the count so new
        # data adapts the statistics within a few updates.
        _obs_decay = _jax_kw_resume.get("obs_rms_decay_on_resume", 0.01)
        if _obs_decay < 1.0:
            _old_count = obs_rms.count
            obs_rms = decay_running_stats(obs_rms, decay_factor=_obs_decay)
            print(f"  obs_rms count decayed: {_old_count:,.0f} → {obs_rms.count:,.0f} (factor={_obs_decay})")
    print(f"Resumed from {_ckpt_path} (update {_resume_update})")
    if "reward_history" in _ckpt:
        print(f"  Prior history: {len(_ckpt['reward_history'])} updates, best reward: {max(_ckpt['reward_history']):.4f}")

n_params = sum(p.size for p in jax.tree.leaves(params))
print(f"ActorCritic parameters: {n_params:,}")

## 5. PPO Implementation
Core PPO functions (`sample_action`, `compute_gae`, `ppo_loss`) are now
imported from `environments.shared.jax_ppo`. Notebook-specific wrappers
below adapt them for the training loop.

In [ ]:
# PPO wrapper functions — thin bindings to shared library
def nb_sample_action(params, obs, rng):
    return sample_action(params, network, obs, rng)

def nb_ppo_loss(params, obs, actions, old_log_probs, advantages, returns,
                old_values=None, clip_range=0.2, vf_coef=0.5, ent_coef=0.01,
                vf_clip_range=None):
    batch = {"obs": obs, "action": actions, "old_log_prob": old_log_probs,
             "advantage": advantages, "return_": returns}
    if old_values is not None:
        batch["old_value"] = old_values
    config = PPOConfig(clip_range=clip_range, vf_coef=vf_coef, ent_coef=ent_coef,
                       vf_clip_range=vf_clip_range)
    return ppo_loss(params, network, batch, config)

print("PPO functions defined (using shared modules from jax_ppo).")

## 6. Training Loop

In [ ]:
# ---------- Hyperparameters (loaded from TOML [jax] section) ----------

# JAX/MJX training hyperparameters (from TOML, with notebook defaults as fallback)
NUM_ENVS = _jax_kw.get("num_envs", 2048)
ROLLOUT_LEN = _jax_kw.get("rollout_len", 64)
NUM_UPDATES = _jax_kw.get("num_updates", 500)
PPO_EPOCHS = _jax_kw.get("ppo_epochs", 4)
MINIBATCH_SIZE = _jax_kw.get("minibatch_size", 512)
LEARNING_RATE = _jax_kw.get("learning_rate", 3e-4)
MAX_GRAD_NORM = _jax_kw.get("max_grad_norm", 0.5)
GAMMA = _jax_kw.get("gamma", 0.99)
GAE_LAMBDA = _jax_kw.get("gae_lambda", 0.95)
CLIP_RANGE = _jax_kw.get("clip_range", 0.2)
ENT_COEF = _jax_kw.get("ent_coef", 0.01)
FALL_PENALTY = _jax_kw.get("fall_penalty", -10.0)
RESET_NOISE_SCALE = _jax_kw.get("reset_noise_scale", 0.05)
INIT_QPOS_NOISE = _jax_kw.get("init_qpos_noise", 0.01)
INIT_YAW_NOISE = _jax_kw.get("init_yaw_noise", 0.1)

# Curriculum warmup (constrains policy updates while critic adapts to new reward landscape)
WARMUP_UPDATES = _jax_kw.get("warmup_updates", 0)          # 0 = no warmup (Stage 1 default)
WARMUP_CLIP_RANGE = _jax_kw.get("warmup_clip_range", 0.02)
WARMUP_ENT_COEF = _jax_kw.get("warmup_ent_coef", 0.02)

# Reward ramp (linearly ramp a reward weight from a fraction to its full value)
RAMP_UPDATES = _jax_kw.get("ramp_updates", 0)              # 0 = no ramp (Stage 1 default)
RAMP_ATTR = _jax_kw.get("ramp_attr", "forward_vel_weight")
RAMP_START_FRACTION = _jax_kw.get("ramp_start_fraction", 0.1)

print("Training config (from TOML [jax] section):")
print(f"  Species: {SPECIES}")
print(f"  Envs: {NUM_ENVS}")
print(f"  Rollout length: {ROLLOUT_LEN}")
print(f"  Updates: {NUM_UPDATES}")
print(f"  Total env steps: {NUM_ENVS * ROLLOUT_LEN * NUM_UPDATES:,}")
print(f"  Stage: {CURRENT_STAGE} ({ctx.stage_name})")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Gamma: {GAMMA}")
print(f"  Clip range: {CLIP_RANGE}")
print(f"  Entropy coef: {ENT_COEF}")
print(f"  Max grad norm: {MAX_GRAD_NORM}")
print(f"  Fall penalty: {FALL_PENALTY}")
print(f"  Reset noise: joints={RESET_NOISE_SCALE}, xy={INIT_QPOS_NOISE}, yaw={INIT_YAW_NOISE}")
if WARMUP_UPDATES > 0:
    print(f"  Warmup: {WARMUP_UPDATES} updates (clip={WARMUP_CLIP_RANGE}, ent={WARMUP_ENT_COEF})")
if RAMP_UPDATES > 0:
    print(f"  Reward ramp: {RAMP_ATTR} from {RAMP_START_FRACTION:.0%} to 100% over {RAMP_UPDATES} updates")
print(f"  Reward config: {reward_cfg}")

In [ ]:
# Reward and termination wrappers already created by make_reward_fns() above.
# compute_reward, compute_reward_detailed, is_terminated are ready to use.
print(f"Active reward components: {[k for k, v in reward_cfg.items() if isinstance(v, (int, float)) and v != 0]}")

In [ ]:
# Initialize environments using MJXDinoEnv.reset()
# Use a different seed from network init (PRNGKey(42)) to avoid correlated streams
rng = jax.random.PRNGKey(0)
rng, reset_rng = jax.random.split(rng)
states = env.reset(reset_rng)

print(f"Initialized {NUM_ENVS} parallel environments via MJXDinoEnv.reset().")
print(f"Obs shape: {states.obs.shape}")
print(f"Reset noise: joints=±{env.config.reset_noise_scale}, "
      f"xy=±{env.config.init_qpos_noise}m, yaw=±{env.config.init_yaw_noise}rad")

In [ ]:
# Optimizer (with gradient clipping and LR decay)
# Read learning_rate_end from TOML [jax] section for linear LR schedule
LEARNING_RATE_END = _jax_kw.get("learning_rate_end", None)
VF_CLIP_RANGE = _jax_kw.get("vf_clip_range", None)
TARGET_KL = _jax_kw.get("target_kl", 0.05)

optimizer = make_optimizer(PPOConfig(
    learning_rate=LEARNING_RATE,
    learning_rate_end=LEARNING_RATE_END,
    max_grad_norm=MAX_GRAD_NORM,
    total_updates=NUM_UPDATES,
    n_epochs=PPO_EPOCHS,
))
opt_state = optimizer.init(params)

if LEARNING_RATE_END is not None:
    print(f"LR schedule: {LEARNING_RATE} -> {LEARNING_RATE_END} (linear decay over {NUM_UPDATES * PPO_EPOCHS} steps)")
if VF_CLIP_RANGE is not None:
    print(f"Value function clipping: +/- {VF_CLIP_RANGE}")
if TARGET_KL is not None:
    print(f"KL early stopping: target_kl={TARGET_KL}")


# JIT-compiled PPO update step (with gradient norm and loss decomposition)
@jax.jit
def ppo_update(params, opt_state, obs, actions, log_probs, advantages, returns,
               old_values=None, clip_range=CLIP_RANGE, ent_coef=ENT_COEF,
               vf_clip_range=VF_CLIP_RANGE):
    (loss, aux), grads = jax.value_and_grad(nb_ppo_loss, has_aux=True)(
        params,
        obs,
        actions,
        log_probs,
        advantages,
        returns,
        old_values=old_values,
        clip_range=clip_range,
        ent_coef=ent_coef,
        vf_clip_range=vf_clip_range,
    )
    grad_norm = optax.global_norm(grads)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss, aux, grad_norm


# JIT-compiled batched action sampling
@jax.jit
def batched_sample(params, obs_batch, rng):
    rngs = jax.random.split(rng, obs_batch.shape[0])
    return jax.vmap(nb_sample_action, in_axes=(None, 0, 0))(params, obs_batch, rngs)




# Reward component diagnostics (uses env state's embedded MJX data)
@jax.jit
def batched_reward_components(states, action_batch):
    """Per-component reward breakdown across all envs (for diagnostics)."""
    return jax.vmap(compute_reward_detailed, in_axes=(0, 0, None))(states.data, action_batch, reward_cfg)


# ==========================================================================
# OPTIMISED: Scanned PPO update epochs (with KL early stopping)
# ==========================================================================
# Replaces the Python double-loop (PPO_EPOCHS x n_minibatches) with nested
# jax.lax.scan, compiling the entire PPO update into one XLA program.
# KL early stopping: when approx_kl exceeds TARGET_KL, remaining epochs
# are skipped (params/opt_state frozen via jnp.where).

@jax.jit
def scan_ppo_epochs(params, opt_state, flat_obs, flat_act, flat_lp, flat_adv,
                    flat_ret, flat_val, rng, clip_range, ent_coef):
    """Run PPO_EPOCHS of minibatch gradient updates with KL early stopping."""
    total_samples = flat_obs.shape[0]
    n_minibatches = total_samples // MINIBATCH_SIZE

    def epoch_fn(carry, _):
        params, opt_state, rng, kl_exceeded = carry
        rng, rng_perm = jax.random.split(rng)
        perm = jax.random.permutation(rng_perm, total_samples)

        def to_mbs(arr):
            return arr[perm[:n_minibatches * MINIBATCH_SIZE]].reshape(
                n_minibatches, MINIBATCH_SIZE, *arr.shape[1:])

        mb_data = (to_mbs(flat_obs), to_mbs(flat_act), to_mbs(flat_lp),
                   to_mbs(flat_adv), to_mbs(flat_ret), to_mbs(flat_val))

        def mb_step(carry, mb):
            params, opt_state, kl_exceeded = carry
            obs, act, lp, adv, ret, val = mb
            new_params, new_opt_state, loss, aux, gn = ppo_update(
                params, opt_state, obs, act, lp, adv, ret,
                old_values=val, clip_range=clip_range, ent_coef=ent_coef)

            # KL early stopping: freeze params if KL exceeded
            approx_kl = aux["approx_kl"]
            use_target_kl = TARGET_KL is not None
            kl_over = use_target_kl & (approx_kl > TARGET_KL)
            should_skip = kl_exceeded | kl_over

            out_params = jax.tree.map(
                lambda new, old: jnp.where(should_skip, old, new),
                new_params, params)
            out_opt_state = jax.tree.map(
                lambda new, old: jnp.where(should_skip, old, new) if hasattr(new, "shape") else new,
                new_opt_state, opt_state)

            return (out_params, out_opt_state, should_skip), (loss, aux, gn)

        (params, opt_state, kl_exceeded), (losses, auxs, gns) = jax.lax.scan(
            mb_step, (params, opt_state, kl_exceeded), mb_data)
        return (params, opt_state, rng, kl_exceeded), (losses, auxs, gns)

    init_carry = (params, opt_state, rng, jnp.bool_(False))
    (params, opt_state, _, _), (all_losses, all_auxs, all_gns) = jax.lax.scan(
        epoch_fn, init_carry, None, length=PPO_EPOCHS)

    mean_loss = jnp.mean(all_losses)
    mean_gn = jnp.mean(all_gns)
    mean_aux = jax.tree.map(jnp.mean, all_auxs)
    return params, opt_state, mean_loss, mean_aux, mean_gn


print("Training functions compiled (using shared optax optimizer).")
print("  rollout: Python loop with async GPU dispatch (physics-bound)")
print("  [optimised] scan_ppo_epochs: fused PPO updates via jax.lax.scan")
print("  [fix] KL early stopping enabled in scan_ppo_epochs")
print("  [fix] Value function clipping enabled via old_values + vf_clip_range")
print("  [fix] LR decay enabled via learning_rate_end")


In [ ]:
# ---------- Main Training Loop ----------
# Rollout: Python loop with async GPU dispatch (physics is the bottleneck,
# not dispatch overhead — scan added 281s compile time for 0% speedup).
# PPO updates: jax.lax.scan (compiles fast, gives real speedup).

from environments.shared.jax_checkpoint import CheckpointManager, save_checkpoint
from environments.shared.jax_training_utils import (
    StabilityMonitor,
    TrainingCSVLogger,
    RolloutProfiler,
    EpisodeStatsAccumulator,
    compute_episode_stats,
)
from environments.shared.reporting import format_duration

CHECKPOINT_FREQ = 25  # Save checkpoint every N updates
MAX_CHECKPOINTS = 5   # Keep only the last N checkpoints (saves disk space)

# Console log frequency based on VERBOSE:
# 0 = only final summary, 1 = every 20 updates (default), 2 = every update
_LOG_INTERVAL = {0: None, 1: 20, 2: 1}.get(VERBOSE, 20)

reward_history = []
loss_history = []
diagnostics_history = []
reward_component_history = []
episode_return_history = []

# Best model tracking
best_reward = -float("inf")
best_params = None
best_update = -1

# Episode return accumulators (on-device — no host syncs during rollout)
_ep_returns = jnp.zeros(NUM_ENVS)
_ep_lengths = jnp.zeros(NUM_ENVS, dtype=jnp.int32)

# Episode stats accumulator — persists across rollouts so that episodes
# spanning multiple 64-step rollouts are tracked correctly.
_ep_stats_acc = EpisodeStatsAccumulator()

# --- Library utilities (replace inline implementations) ---
_ckpt_mgr = CheckpointManager(MODEL_DIR, prefix="checkpoint", max_keep=MAX_CHECKPOINTS)
_stability = StabilityMonitor()
_profiler = RolloutProfiler(interval=50)
_csv_logger = TrainingCSVLogger(OUTPUT_DIR / "training_log.csv")
csv_path = _csv_logger.path  # Expose for downstream cells

# Warmup/ramp state
_warmup_active = WARMUP_UPDATES > 0
_ramp_active = RAMP_UPDATES > 0
_ramp_target_value = reward_cfg.get(RAMP_ATTR, 0.0) if _ramp_active else 0.0

if _warmup_active:
    _original_clip_range = CLIP_RANGE
    _original_ent_coef = ENT_COEF

# ==================== GPU / Device Diagnostics ====================
_device = jax.devices()[0]
_dev_kind = _device.device_kind if hasattr(_device, 'device_kind') else str(_device.platform).upper()
print(f"Device:     {_device} ({_dev_kind})")
print(f"JAX:        {jax.__version__}")
try:
    _mem_stats = _device.memory_stats()
    if _mem_stats:
        _total_gb = _mem_stats.get('bytes_limit', 0) / 1e9
        _used_gb  = _mem_stats.get('peak_bytes_in_use', _mem_stats.get('bytes_in_use', 0)) / 1e9
        print(f"GPU memory: {_used_gb:.1f} / {_total_gb:.1f} GB used")
except Exception:
    print("GPU memory: (stats unavailable)")
_batch_size_total = ROLLOUT_LEN * NUM_ENVS
print(f"Batch size: {_batch_size_total:,} ({ROLLOUT_LEN} steps x {NUM_ENVS} envs)")
print(f"PPO:        {PPO_EPOCHS} epochs x {_batch_size_total // MINIBATCH_SIZE} minibatches of {MINIBATCH_SIZE}")
_total_env_steps = NUM_UPDATES * ROLLOUT_LEN * NUM_ENVS
print(f"Total:      {_total_env_steps:,} env steps over {NUM_UPDATES} updates")

_start_update = _resume_update
print(f"\nStarting training: updates {_start_update}..{_start_update + NUM_UPDATES - 1} "
      f"({ROLLOUT_LEN} steps x {NUM_ENVS} envs)")
print(f"Checkpoint frequency: every {CHECKPOINT_FREQ} updates (keep last {MAX_CHECKPOINTS})")
if _warmup_active:
    print(f"Warmup: updates 0..{WARMUP_UPDATES - 1} (clip_range={WARMUP_CLIP_RANGE}, ent_coef={WARMUP_ENT_COEF})")
if _ramp_active:
    print(f"Reward ramp: {RAMP_ATTR} from {_ramp_target_value * RAMP_START_FRACTION:.4f} to {_ramp_target_value:.4f} over updates 0..{RAMP_UPDATES - 1}")
print(f"CSV log: {csv_path}")
print("=" * 70)

t_start = time.time()
_cum_t_rollout = 0.0
_cum_t_ppo = 0.0
_compile_time = 0.0

try:
    for update in range(_start_update, _start_update + NUM_UPDATES):
        # ---------- Warmup ----------
        relative_update = update - _start_update
        if _warmup_active:
            if relative_update < WARMUP_UPDATES:
                _active_clip_range = WARMUP_CLIP_RANGE
                _active_ent_coef = WARMUP_ENT_COEF
            else:
                _active_clip_range = _original_clip_range
                _active_ent_coef = _original_ent_coef
                if relative_update == WARMUP_UPDATES and (_LOG_INTERVAL is not None):
                    print(f"  >>> Warmup complete at update {update}: restoring clip_range={_original_clip_range}, ent_coef={_original_ent_coef}")
        else:
            _active_clip_range = CLIP_RANGE
            _active_ent_coef = ENT_COEF

        # ---------- Reward ramp ----------
        if _ramp_active:
            if relative_update < RAMP_UPDATES:
                ramp_progress = relative_update / RAMP_UPDATES
                ramp_value = _ramp_target_value * (RAMP_START_FRACTION + (1.0 - RAMP_START_FRACTION) * ramp_progress)
            else:
                ramp_value = _ramp_target_value
            reward_cfg[RAMP_ATTR] = ramp_value

        # ---------- Collect rollout (MJXDinoEnv handles step+obs+reward+reset) ----------
        _t_phase = time.time()
        all_obs, all_actions, all_log_probs, all_values = [], [], [], []
        all_rewards, all_dones, all_full_dones = [], [], []

        for t in range(ROLLOUT_LEN):
            rng, rng_act, rng_step = jax.random.split(rng, 3)

            # Observe current state and sample action
            obs = normalize_obs(states.obs, obs_rms)
            actions, log_probs, values = batched_sample(params, obs, rng_act)

            # Step: physics + obs + reward + termination + auto-reset (all JIT-compiled)
            states, rewards, terminated, truncated = env.step(states, actions, rng_step)
            dones = terminated | truncated

            # Episode tracking (fully on-device, no host syncs)
            _ep_returns = _ep_returns + rewards
            _ep_lengths = _ep_lengths + 1

            # For GAE: only zero bootstrap on true termination, not truncation.
            # Truncated episodes still have value — the agent didn't fail, it ran out of time.
            gae_dones = terminated

            all_obs.append(obs)
            all_actions.append(actions)
            all_log_probs.append(log_probs)
            all_values.append(values)
            all_rewards.append(rewards)
            all_dones.append(gae_dones.astype(jnp.float32))
            all_full_dones.append(dones.astype(jnp.float32))

            # Reset episode accumulators for done envs (states already auto-reset by env)
            _ep_returns = jnp.where(dones, 0.0, _ep_returns)
            _ep_lengths = jnp.where(dones, 0, _ep_lengths)

        # Stack rollout data
        obs_t = jnp.stack(all_obs)
        act_t = jnp.stack(all_actions)
        lp_t = jnp.stack(all_log_probs)
        val_t = jnp.stack(all_values)
        rew_t = jnp.stack(all_rewards)
        done_t = jnp.stack(all_dones)

        # Block for accurate timing
        jax.block_until_ready(done_t)
        _t_rollout = time.time() - _t_phase
        _cum_t_rollout += _t_rollout

        # Rollout timing (env.step bundles all ops, no per-op breakdown)

        # ---------- Episode stats (using library helper) ----------
        full_done_t = jnp.stack(all_full_dones)
        full_done_np = np.array(full_done_t)
        rew_np = np.array(rew_t)
        fall_rate = float(full_done_np.sum()) / (ROLLOUT_LEN * NUM_ENVS)
        _completed_returns, _completed_lengths = compute_episode_stats(rew_np, full_done_np, _ep_stats_acc)

        # ---------- Update obs normalisation (once per update) ----------
        obs_batch_flat = obs_t.reshape(-1, OBS_DIM)
        obs_rms = update_running_stats(obs_rms, obs_batch_flat)

        # ---------- Bootstrap value for GAE ----------
        rng, rng_bootstrap = jax.random.split(rng)
        obs_final = normalize_obs(states.obs, obs_rms)
        _, _, bootstrap_values = batched_sample(params, obs_final, rng_bootstrap)
        val_t_plus1 = jnp.concatenate([val_t, bootstrap_values[None]], axis=0)

        # ---------- Compute advantages ----------
        advantages, returns = compute_gae(rew_t, val_t_plus1, done_t, GAMMA, GAE_LAMBDA)

        # Flatten: (T * N, ...)
        flat_obs = obs_t.reshape(-1, OBS_DIM)
        flat_act = act_t.reshape(-1, ACT_DIM)
        flat_lp = lp_t.reshape(-1)
        flat_adv = advantages.reshape(-1)
        flat_ret = returns.reshape(-1)
        flat_val = val_t.reshape(-1)

        # ---------- PPO update (fused scan — compiles fast, real speedup) ----------
        _t_phase = time.time()
        rng, rng_ppo = jax.random.split(rng)
        params, opt_state, avg_loss, avg_aux, avg_grad_norm = scan_ppo_epochs(
            params, opt_state, flat_obs, flat_act, flat_lp, flat_adv, flat_ret,
            flat_val, rng_ppo, jnp.float32(_active_clip_range), jnp.float32(_active_ent_coef),
        )

        # Transfer scalar metrics to host
        avg_loss = float(avg_loss)
        avg_grad_norm = float(avg_grad_norm)
        avg_aux = {k: float(v) for k, v in avg_aux.items()}
        _t_ppo = time.time() - _t_phase
        _cum_t_ppo += _t_ppo

        avg_reward = float(rew_t.mean())

        # Episode return stats
        if _completed_returns:
            mean_ep_return = np.mean(_completed_returns)
            mean_ep_length = np.mean(_completed_lengths)
        else:
            mean_ep_return = float("nan")
            mean_ep_length = float("nan")
        episode_return_history.append(mean_ep_return)

        reward_history.append(avg_reward)
        loss_history.append(avg_loss)
        diagnostics_history.append({
            "reward": avg_reward,
            "episode_return": mean_ep_return,
            "episode_length": mean_ep_length,
            "loss": avg_loss,
            "grad_norm": avg_grad_norm,
            "fall_rate": fall_rate,
            "t_rollout": _t_rollout,
            "t_ppo": _t_ppo,
            **avg_aux,
        })

        # Per-component reward diagnostics (sample every 10 updates)
        if relative_update % 10 == 0:
            try:
                _comp = batched_reward_components(states, all_actions[-1])
                _comp_means = {k: float(jnp.mean(v)) for k, v in _comp.items()}
                _comp_means["update"] = update
                reward_component_history.append(_comp_means)
            except Exception:
                pass  # Don't break training if diagnostics fail

        # ==================== Stability watchdog ====================
        _kl = avg_aux.get("approx_kl", 0.0)
        should_halt, _is_unstable, _stab_msg = _stability.check(
            _kl, avg_grad_norm, avg_loss, update
        )
        if _stab_msg:
            print(f"  {_stab_msg}")
        if should_halt:
            break

        # Track best model
        _track_metric = mean_ep_return if not np.isnan(mean_ep_return) else avg_reward
        if _track_metric > best_reward and not _is_unstable:
            best_reward = _track_metric
            best_params = jax.device_get(params)
            best_update = update

        # CSV log
        elapsed = time.time() - t_start
        steps_done = (update - _start_update + 1) * ROLLOUT_LEN * NUM_ENVS
        sps = steps_done / elapsed
        _csv_logger.log({
            "update": update,
            "reward_per_step": f"{avg_reward:.4f}",
            "episode_return": f"{mean_ep_return:.2f}" if not np.isnan(mean_ep_return) else "",
            "episode_length": f"{mean_ep_length:.1f}" if not np.isnan(mean_ep_length) else "",
            "total_loss": f"{avg_loss:.4f}",
            "policy_loss": f"{avg_aux['policy_loss']:.4f}",
            "value_loss": f"{avg_aux['value_loss']:.4f}",
            "entropy": f"{avg_aux['entropy']:.4f}",
            "approx_kl": f"{avg_aux['approx_kl']:.6f}",
            "clip_fraction": f"{avg_aux['clip_fraction']:.4f}",
            "grad_norm": f"{avg_grad_norm:.4f}",
            "mean_std": f"{avg_aux['mean_std']:.4f}",
            "steps": steps_done,
            "sps": f"{sps:.0f}",
            "fall_rate": f"{fall_rate:.4f}",
            "elapsed": f"{elapsed:.1f}",
            "t_rollout": f"{_t_rollout:.3f}",
            "t_ppo": f"{_t_ppo:.3f}",
        })

        # Console logging
        if _LOG_INTERVAL is not None and (
            (update - _start_update) % _LOG_INTERVAL == 0
            or update == _start_update + NUM_UPDATES - 1
        ):
            updates_done = update - _start_update + 1
            updates_left = NUM_UPDATES - updates_done
            eta = (elapsed / updates_done) * updates_left if updates_done > 0 else 0
            eta_str = f"{eta / 60:.0f}m" if eta > 60 else f"{eta:.0f}s"
            ep_ret_str = f"ep_ret={mean_ep_return:+.1f}" if not np.isnan(mean_ep_return) else "ep_ret=n/a"
            print(
                f"[{update:4d}/{_start_update + NUM_UPDATES}]  "
                f"r/step={avg_reward:+.3f}  {ep_ret_str}  "
                f"loss={avg_loss:.4f}  "
                f"pi={avg_aux['policy_loss']:.3f}  v={avg_aux['value_loss']:.3f}  "
                f"ent={avg_aux['entropy']:.3f}  kl={avg_aux['approx_kl']:.4f}  "
                f"grad={avg_grad_norm:.3f}  falls={fall_rate:.1%}  "
                f"SPS={sps:,.0f}  ETA={eta_str}  "
                f"[{_t_rollout:.1f}s+{_t_ppo:.2f}s]"
            )

        # Periodic checkpointing (with rotation via CheckpointManager)
        if (update - _start_update + 1) % CHECKPOINT_FREQ == 0:
            _ckpt_mgr.save(
                params, update + 1, obs_rms=obs_rms,
                history={
                    "reward": reward_history,
                    "loss": loss_history,
                    "episode_return": episode_return_history,
                },
            )
            if _LOG_INTERVAL is not None:
                print(f"  >>> Checkpoint saved: {_ckpt_mgr.latest}")

finally:
    _csv_logger.close()

# ==================== Training Summary ====================
elapsed = time.time() - t_start
actual_updates = update - _start_update + 1  # handles early halt
total_steps = actual_updates * ROLLOUT_LEN * NUM_ENVS
print("=" * 70)
print(f"Done! {total_steps:,} steps in {format_duration(elapsed)} ({total_steps / elapsed:,.0f} SPS)")
print(f"Best metric: {best_reward:+.4f} at update {best_update}")

# Timing breakdown
_pct_rollout = 100 * _cum_t_rollout / elapsed if elapsed > 0 else 0
_pct_ppo     = 100 * _cum_t_ppo / elapsed if elapsed > 0 else 0
_pct_other   = 100 - _pct_rollout - _pct_ppo
print(f"\nTiming breakdown:")
print(f"  Rollout (MJX physics): {_cum_t_rollout:7.1f}s  ({_pct_rollout:4.1f}%)")
print(f"  PPO updates:           {_cum_t_ppo:7.1f}s  ({_pct_ppo:4.1f}%)")
print(f"  Other (GAE/IO/log):    {elapsed - _cum_t_rollout - _cum_t_ppo:7.1f}s  ({_pct_other:4.1f}%)")

# Physics throughput
_physics_steps = actual_updates * ROLLOUT_LEN * NUM_ENVS * FRAME_SKIP
if _cum_t_rollout > 0:
    print(f"\nMJX physics: {_physics_steps:,} steps in {_cum_t_rollout:.1f}s "
          f"({_physics_steps / _cum_t_rollout:,.0f} physics steps/sec)")

# Reward trend
if len(reward_history) >= 10:
    _first10 = np.mean(reward_history[:10])
    _last10 = np.mean(reward_history[-10:])
    _trend = "improved" if _last10 > _first10 else "declined"
    print(f"\nReward trend: {_first10:.3f} (first 10) -> {_last10:.3f} (last 10)  [{_trend}]")

if _stability.total_warnings > 0:
    print(f"\nTotal stability warnings: {_stability.total_warnings}")

# Per-operation profiling summary
_prof_summary = _profiler.summary()
if _prof_summary:
    print(f"\n{_prof_summary}")

# GPU memory after training
try:
    _mem_stats = _device.memory_stats()
    if _mem_stats:
        _peak_gb = _mem_stats.get('peak_bytes_in_use', 0) / 1e9
        print(f"Peak GPU memory: {_peak_gb:.2f} GB")
except Exception:
    pass

# Save final trained parameters
params_path = MODEL_DIR / "params.pkl"
save_checkpoint(
    params_path, params, obs_rms=obs_rms,
    extra={
        "best_params": best_params,
        "best_reward": best_reward,
        "best_update": best_update,
    },
    history={
        "reward": reward_history,
        "loss": loss_history,
        "episode_return": episode_return_history,
        "diagnostics": diagnostics_history,
    },
)
print(f"\nParameters and training history saved to: {params_path}")
print(f"Training log CSV saved to: {csv_path}")

## Stage Gate Evaluation

Run full evaluation episodes on CPU to check whether the curriculum gate
thresholds (min reward and min episode length) have been met. Both conditions
must pass before advancing to the next training stage.

In [ ]:
# ---------- Stage Gate Evaluation (via shared library) ----------
from environments.shared.jax_notebook import run_stage_evaluation, print_eval_summary

eval_results, stage_results, gate_passed, gate_failures = run_stage_evaluation(
    ctx, env, params, network, obs_rms,
    n_episodes=30,
    best_params=best_params,
    best_reward=best_reward,
    best_update=best_update,
    total_steps=total_steps if 'total_steps' in dir() else 0,
    elapsed=elapsed if 'elapsed' in dir() else 0.0,
)

# Expose diagnostic variables for downstream plotting cells
diag_tilt = eval_results.diag_tilt
diag_fwd_vel = eval_results.diag_fwd_vel
diag_pelvis_h = eval_results.diag_pelvis_h
diag_l_foot = eval_results.diag_l_foot
diag_r_foot = eval_results.diag_r_foot
diag_energy = eval_results.diag_energy
diag_reward_components = eval_results.diag_reward_components
frames = getattr(eval_results, 'frames', [])

print_eval_summary(eval_results, gate_passed, gate_failures, CURRENT_STAGE)

## Save Results & Stage Summary

Save structured training artifacts: stage summary text file, collected results
CSV (compatible with sweep analysis tooling), and model checkpoints.

In [ ]:
from environments.shared.reporting import save_jax_stage_artifacts

artifact_paths = save_jax_stage_artifacts(
    species=SPECIES,
    stage=CURRENT_STAGE,
    stage_config=_stage_cfg,
    stage_results=stage_results,
    stage_dir=STAGE_DIR,
    run_dir=RUN_DIR,
    eval_results=eval_results,
    params=jax.device_get(params),
    obs_rms=obs_rms,
    seed=42,
    num_envs=NUM_ENVS,
    reward_cfg=reward_cfg,
    best_params=best_params,
    best_reward=best_reward,
    best_update=best_update,
)

# Print saved artifact paths
for name, path in artifact_paths.items():
    print(f"{name}: {path}")

# Print the stage summary inline
print()
print(artifact_paths["stage_summary"].read_text())

## 7. Training Curves & Locomotion Diagnostics

In [ ]:
# Training curves — uses shared plot_training_curves from jax_viz
curve_path = OUTPUT_DIR / "training_curves.png"

plot_training_curves(
    reward_history=reward_history,
    loss_history=loss_history,
    episode_return_history=episode_return_history,
    diagnostics_history=diagnostics_history,
    species=SPECIES,
    stage=CURRENT_STAGE,
    output_path=curve_path,
    show=True,
)

print(f"Training curves saved to: {curve_path}")

In [ ]:
# Locomotion diagnostics — uses shared plot_locomotion_diagnostics from jax_viz

plot_locomotion_diagnostics(
    eval_results,
    species=SPECIES,
    stage=CURRENT_STAGE,
    max_tilt_angle=MAX_TILT_ANGLE,
    healthy_z_range=(HEALTHY_Z_MIN, HEALTHY_Z_MAX),
    output_dir=STAGE_DIR,
    show=True,
)

print(f"Locomotion diagnostics saved to: {STAGE_DIR}")

### Reward Component Diagnostics

Per-component reward breakdown sampled during training.  
Shows which reward terms the agent is exploiting vs ignoring, plus body state variables vs termination thresholds.

In [ ]:
# Reward component breakdown over training
# Shows per-component reward contributions to diagnose exploit poses

if reward_component_history:
    import matplotlib.pyplot as plt
    import numpy as np

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f"Reward Component Diagnostics — {SPECIES} Stage {CURRENT_STAGE}", fontsize=14)

    updates = [d["update"] for d in reward_component_history]

    # --- Plot 1: Per-component reward curves ---
    ax = axes[0, 0]
    reward_keys = [k for k in reward_component_history[0] if not k.startswith("_") and k != "update"]
    for key in reward_keys:
        vals = [d.get(key, 0.0) for d in reward_component_history]
        ax.plot(updates, vals, label=key, linewidth=1.5)
    ax.set_xlabel("Update")
    ax.set_ylabel("Mean reward component")
    ax.set_title("Per-component rewards")
    ax.legend(fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)

    # --- Plot 2: Stacked area chart ---
    ax = axes[0, 1]
    pos_keys = [k for k in reward_keys if any(d.get(k, 0) > 0 for d in reward_component_history)]
    neg_keys = [k for k in reward_keys if any(d.get(k, 0) < 0 for d in reward_component_history)]
    for key in pos_keys:
        vals = [d.get(key, 0.0) for d in reward_component_history]
        ax.fill_between(updates, 0, vals, alpha=0.4, label=key)
    for key in neg_keys:
        vals = [d.get(key, 0.0) for d in reward_component_history]
        ax.fill_between(updates, 0, vals, alpha=0.4, label=key)
    ax.set_xlabel("Update")
    ax.set_ylabel("Reward")
    ax.set_title("Reward composition")
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.3)

    # --- Plot 3: State diagnostics (pelvis_z, forward_z, foot_contact) ---
    ax = axes[1, 0]
    if "_pelvis_z" in reward_component_history[0]:
        ax.plot(updates, [d["_pelvis_z"] for d in reward_component_history], label="pelvis_z", color="blue")
        ax.axhline(y=HEALTHY_Z_MIN, color="blue", linestyle="--", alpha=0.5, label=f"z_min={HEALTHY_Z_MIN}")
    if "_forward_z" in reward_component_history[0]:
        ax2 = ax.twinx()
        ax2.plot(updates, [d["_forward_z"] for d in reward_component_history], label="forward_z", color="red")
        nd_thresh = NATURAL_FORWARD_Z - _NOSEDIVE_TERMINATION_THRESHOLD
        ax2.axhline(y=nd_thresh, color="red", linestyle="--", alpha=0.5, label=f"nosedive={nd_thresh:.2f}")
        ax2.set_ylabel("forward_z", color="red")
        ax2.legend(loc="lower right", fontsize=8)
    ax.set_xlabel("Update")
    ax.set_ylabel("pelvis_z", color="blue")
    ax.set_title("Body state vs termination thresholds")
    ax.legend(loc="upper left", fontsize=8)
    ax.grid(True, alpha=0.3)

    # --- Plot 4: Foot contact rate ---
    ax = axes[1, 1]
    if "_has_foot_contact" in reward_component_history[0]:
        fc_vals = [d["_has_foot_contact"] for d in reward_component_history]
        ax.plot(updates, fc_vals, label="foot contact rate", color="green", linewidth=2)
        ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel("Update")
    ax.set_ylabel("Foot contact rate")
    ax.set_title("Foot contact (alive bonus gate)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    diag_path = OUTPUT_DIR / "reward_components.png"
    plt.savefig(diag_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved to {diag_path}")

    # Print final component breakdown
    print("\nFinal reward component means:")
    last = reward_component_history[-1]
    total = 0.0
    for k in sorted(reward_keys):
        v = last.get(k, 0.0)
        total += v
        print(f"  {k:20s}: {v:+.4f}")
    print(f"  {'total':20s}: {total:+.4f}")
else:
    print("No reward component data collected. Re-run training with updated code.")


## 8. Record Training Video

Record a video of the best trained policy (highest reward during training) using
the CPU MuJoCo renderer. The JAX policy is evaluated deterministically (using the
action mean).

In [ ]:
try:
    import mediapy  # noqa: F401
    _HAS_MEDIAPY = True
except ImportError:
    _HAS_MEDIAPY = False
    print("mediapy not installed — installing now...")
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "mediapy"])
    import mediapy  # noqa: F401
    _HAS_MEDIAPY = True
    print("mediapy installed successfully.")

if _HAS_MEDIAPY:
    video_params = best_params if best_params is not None else jax.device_get(params)
    print(f"Recording video with best model (update {best_update}, reward {best_reward:+.4f})")

    video_path = str(OUTPUT_DIR / "evaluation.mp4")

    frames, episode_reward = record_training_video(
        mj_model, video_params, network, obs_rms,
        get_obs_fn=get_obs,
        normalize_obs_fn=normalize_obs,
        scale_action_fn=scale_action,
        reward_fn=compute_reward,
        reward_cfg=reward_cfg,
        max_episode_steps=ctx.max_episode_steps,
        frame_skip=ctx.frame_skip,
        root_body_id=ctx.root_body_id,
        healthy_z_range=ctx.healthy_z_range,
        max_tilt_angle=ctx.max_tilt_angle,
        natural_forward_z=ctx.natural_forward_z,
        termination_body_heights=ctx.termination_body_heights,
        termination_site_heights=ctx.termination_site_heights,
        success_sites=ctx.success_sites,
        success_threshold=ctx.success_threshold,
        target_body="prey",
        sensor_quat_start=ctx.sensor_layout.quat_start,
        output_path=video_path,
        fps=50,
        camera_track_body=ctx.camera_track_body,
        camera_distance=ctx.camera_distance,
        show=True,
    )

    print(f"Episode reward: {episode_reward:.2f} | {len(frames)} frames")
    print(f"Saved to: {video_path}")

In [ ]:
# Create a frame collage for quick visual review
# Single image with labelled frame numbers and timestamps

collage_path = STAGE_DIR / "eval_collage.png"
collage_fig = create_frame_collage(
    frames,
    output_path=collage_path,
    num_frames=10,
    cols=5,
    title=f"{SPECIES.capitalize()} Stage {CURRENT_STAGE} — Eval Rollout ({len(frames)} frames, {len(frames)/50:.1f}s)",
    fps=50,
    show=True,
)
print(f"Collage saved to: {collage_path}")

# Auto-download in Google Colab
try:
    from google.colab import files
    files.download(str(collage_path))
except ImportError:
    pass

## 9. Next Steps

To continue with curriculum training, change `CURRENT_STAGE` in the configuration
cell above and re-run from there. The reward config is loaded automatically from the
TOML files in `configs/<species>/`:

```python
CURRENT_STAGE = 2  # or 3
```

The policy parameters carry over automatically between stages.
After each stage, re-run the Stage Gate Evaluation cell to verify the curriculum
thresholds are met, then re-run the video recording cell to capture the new behavior.

**Resuming from a checkpoint:** If your Colab session disconnects, set
`RESUME_FROM` in the configuration cell to reload parameters and obs stats:

```python
RESUME_FROM = "checkpoint_100.pkl"
```

To train a different species, change `SPECIES` in the configuration cell and
restart from the beginning.

## 10. Auto-Disconnect (Optional)

Optionally disconnect the Colab runtime after completion to free up resources.
Set `AUTO_DISCONNECT = True` in the configuration cell or toggle below.

In [ ]:
AUTO_DISCONNECT = True  # Set to True to disconnect runtime after training

if AUTO_DISCONNECT:
    import time
    print("Training finished. Disconnecting runtime in 5 seconds...")
    time.sleep(5)
    from google.colab import runtime
    runtime.unassign()
else:
    print("Training finished. Runtime kept alive — remember to disconnect manually when done.")